# 벡터 DB 생성 및 적재를 위한 노트북 파일입니다.

### step1 - 필요한 데이터 경로 가져오기

In [4]:
from pathlib import Path

import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Load the CSV from the notebook parent directory.
csv_path = Path("../museum_final_738.csv")
df = pd.read_csv(csv_path, encoding="utf-8-sig")

#테스트
display(df.head(2))


,museum_id,name,ownership_type,sido,sigungu,address,latitude,longitude,operator_name,manager_name,...,official_verified_field_count,text,text_length,text_sha256,source_doc_count,source_types,source_documents_json,text_spans_json,removed_duplicate_sentence_count,merge_review_flags
0,M0001,박진전쟁기념관,공립,경상남도,창녕군,경상남도 창녕군 남지읍 월상길 27,35.447968,128.386193,경상남도 창녕군청,경상남도 창녕군청,...,0.0,박진전쟁기념관은 6·25전쟁 최후의 보루였던 박진전투 현장을 재현하여 민족 분단의 ...,407,1fcf9c09a671bb09df9f0a8d84edf4d13a2fd68b08c0c4...,1,TOUR_API,"[{""doc_id"":""DOC-B0001-TOUR_API-2785601"",""sourc...","[{""start"":0,""end"":94,""doc_ids"":[""DOC-B0001-TOU...",0,NaN
1,M0003,창녕박물관,공립,경상남도,창녕군,경상남도 창녕군 창녕읍 창밀로 34,35.545406,128.505441,경상남도 창녕군청,경상남도 창녕군청,...,0.0,창녕박물관은 1996년 창녕지역 고분군에서 출토된 유물들을 전시할 목적으로 창녕유물...,608,35480738914dd66b95a1bef1f432ae7ce951e01ee9503d...,1,TOUR_API,"[{""doc_id"":""DOC-B0003-TOUR_API-130107"",""source...","[{""start"":0,""end"":60,""doc_ids"":[""DOC-B0003-TOU...",0,NaN


### step2 - 임베딩 모델 가져오기(text-embedding-3-small)

In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

env_path = next((parent / ".env" for parent in [Path.cwd(), *Path.cwd().parents] if (parent / ".env").is_file()), None)
if env_path:
    load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in .env or the environment.")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


### step3 - 벡터DB에 필요한 열만 남기기

In [6]:
# ==== ChromaDB 적재용 데이터 준비 ====

# 1. 사용할 열 선택
chroma_columns = [
    # 식별자 및 임베딩 본문
    "museum_id",
    "text",

    # 시설 기본 정보
    "name",
    "ownership_type",
    "sido",
    "sigungu",
    "address",
    "latitude",
    "longitude",
    "homepage_url",

    # 관람료 및 조건 검색
    "adult_fee",
    "youth_fee",
    "child_fee",
    "adult_fee_status",
    "fee_notes",

    # 운영 정보
    "weekday_open",
    "weekday_close",
    "holiday_open",
    "holiday_close",
    "closure_rule_raw",

    # 편의시설 및 안내
    "amenities_raw",
    "transport_notes",
    "image_url",
    "image_copyright",

    # 출처 및 품질 관리
    "source_types",
    "requires_manual_review",
]

# 2. 필요한 열만 추출 (원본 df 유지)
df_chroma = df[chroma_columns].copy()

# 3. 준비된 데이터 저장
df_chroma.to_csv(
    "museum_chroma_ready.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. 상위 2개 행 확인
display(df_chroma.head(2))

,museum_id,text,name,ownership_type,sido,sigungu,address,latitude,longitude,homepage_url,...,weekday_close,holiday_open,holiday_close,closure_rule_raw,amenities_raw,transport_notes,image_url,image_copyright,source_types,requires_manual_review
0,M0001,박진전쟁기념관은 6·25전쟁 최후의 보루였던 박진전투 현장을 재현하여 민족 분단의 ...,박진전쟁기념관,공립,경상남도,창녕군,경상남도 창녕군 남지읍 월상길 27,35.447968,128.386193,https://www.cng.go.kr/01656/03204/03205.web,...,18:00,09:00,18:00,매주 월요일+1월 1일+설날+추석날,장애인화장실+유모차/휠체어 이용가능,NaN,https://tong.visitkorea.or.kr/cms/resource/99/...,Type1,TOUR_API,True
1,M0003,창녕박물관은 1996년 창녕지역 고분군에서 출토된 유물들을 전시할 목적으로 창녕유물...,창녕박물관,공립,경상남도,창녕군,경상남도 창녕군 창녕읍 창밀로 34,35.545406,128.505441,https://www.cng.go.kr/01656/01671.web,...,18:00,09:00,18:00,매주 월요일+1월 1일+설날+추석날,장애인화장실+유모차/휠체어 대여+유모차/휠체어 이용가능,NaN,https://tong.visitkorea.or.kr/cms/resource/15/...,Type1,TOUR_API,True


### step4 - text 열 임베딩 처리하기 + 적재

- museum_id만 메타데이터가 아니라 ChromaDB의 고유 ID로 저장 (ids로 저장됨)

In [7]:
# ==== ChromaDB 임베딩 및 영구 저장 ====

from pathlib import Path
import pandas as pd
import numpy as np
import chromadb

# 1. 프로젝트 루트 자동 탐색
NOTEBOOK_REL = Path(
    "src/for_build/build_vertorDB/vectoDB_builder.ipynb"
)

cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents]
     if (p / NOTEBOOK_REL).is_file()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"프로젝트 루트를 찾을 수 없습니다. 현재 경로: {cwd}"
    )

# 2. ChromaDB 저장 위치
DB_PATH = PROJECT_ROOT / "data" / "vector_db" / "museum_chroma"
COLLECTION_NAME = "museum_documents"

DB_PATH.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=str(DB_PATH))
collection = client.get_or_create_collection(
    name=COLLECTION_NAME
)

# 3. 데이터 준비
df_ready = df_chroma.copy()

assert df_ready["museum_id"].notna().all()
assert df_ready["museum_id"].is_unique
assert df_ready["text"].notna().all()
assert df_ready["text"].apply(
    lambda x: isinstance(x, str) and bool(x.strip())
).all()

ids = df_ready["museum_id"].astype(str).tolist()
documents = df_ready["text"].tolist()

# 4. 메타데이터 변환
NUMERIC_COLS = {
    "latitude", "longitude",
    "adult_fee", "youth_fee", "child_fee"
}

def make_metadata(row):
    metadata = {}

    for col, value in row.items():
        if col in ("museum_id", "text"):
            continue

        if pd.isna(value):
            continue

        if isinstance(value, str):
            value = value.strip()
            if not value:
                continue

        if col in NUMERIC_COLS:
            value = pd.to_numeric(value, errors="coerce")
            if pd.isna(value) or not np.isfinite(value):
                continue
            value = float(value)
            if col not in ("latitude", "longitude") and value.is_integer():
                value = int(value)

        elif col == "requires_manual_review":
            if isinstance(value, str):
                if value.lower() not in ("true", "false"):
                    continue
                value = value.lower() == "true"
            else:
                value = bool(value)

        elif isinstance(value, (bool, np.bool_)):
            value = bool(value)

        elif isinstance(value, (int, np.integer)):
            value = int(value)

        elif isinstance(value, (float, np.floating)):
            value = float(value)

        else:
            value = str(value)

        metadata[col] = value

    return metadata

metadatas = [
    make_metadata(row)
    for _, row in df_ready.iterrows()
]

# 5. 기존 적재 데이터 확인
existing = collection.get(
    ids=ids,
    include=["documents", "metadatas"]
)

existing_map = {
    doc_id: (doc, meta or {})
    for doc_id, doc, meta in zip(
        existing["ids"],
        existing["documents"],
        existing["metadatas"]
    )
}

# 새 문서 또는 본문 변경
embed_idx = [
    i for i, doc_id in enumerate(ids)
    if doc_id not in existing_map
    or existing_map[doc_id][0] != documents[i]
]

# 본문은 같고 메타데이터만 변경
metadata_idx = [
    i for i, doc_id in enumerate(ids)
    if doc_id in existing_map
    and existing_map[doc_id][0] == documents[i]
    and existing_map[doc_id][1] != metadatas[i]
]

# 6. 필요한 문서만 OpenAI 임베딩
if embed_idx:
    vectors = embeddings.embed_documents(
        [documents[i] for i in embed_idx]
    )

    collection.upsert(
        ids=[ids[i] for i in embed_idx],
        embeddings=vectors,
        documents=[documents[i] for i in embed_idx],
        metadatas=[metadatas[i] for i in embed_idx]
    )

# 7. 메타데이터만 변경된 문서
if metadata_idx:
    collection.update(
        ids=[ids[i] for i in metadata_idx],
        metadatas=[metadatas[i] for i in metadata_idx]
    )

# 8. 저장 결과 확인
print(f"DB 저장 위치: {DB_PATH}")
print(f"Collection: {COLLECTION_NAME}")
print(f"임베딩 생성/갱신: {len(embed_idx)}건")
print(f"메타데이터 갱신: {len(metadata_idx)}건")
print(f"전체 적재 건수: {collection.count()}건")

DB 저장 위치: C:\Users\Playdata\Desktop\mle-01-p3-team1\data\vector_db\museum_chroma
Collection: museum_documents
임베딩 생성/갱신: 738건
메타데이터 갱신: 0건
전체 적재 건수: 738건


### step5 - 저장된 DB 확인

In [8]:
# ==== 저장된 ChromaDB 재조회 및 적재 검증 ====

from IPython.display import display

# 1. 디스크에 저장된 DB에 다시 연결
check_client = chromadb.PersistentClient(
    path=str(DB_PATH)
)

check_collection = check_client.get_collection(
    name=COLLECTION_NAME
)

# 2. 적재 건수 및 ID 검증
stored_count = check_collection.count()

assert stored_count == len(df_ready), (
    f"건수 불일치: {stored_count} / {len(df_ready)}"
)

stored_ids = set(check_collection.get(include=[])["ids"])
assert stored_ids == set(ids), "저장된 museum_id가 원본과 다릅니다."

# 3. 첫 2개 문서와 임베딩 조회
result = check_collection.get(
    ids=ids[:2],
    include=["documents", "metadatas", "embeddings"]
)

assert len(result["ids"]) == 2
assert result["embeddings"] is not None

# 4. 확인용 테이블
check_df = pd.DataFrame([
    {
        "museum_id": doc_id,
        "name": meta.get("name"),
        "text": doc[:100],
        "embedding_dim": len(vector)
    }
    for doc_id, doc, meta, vector in zip(
        result["ids"],
        result["documents"],
        result["metadatas"],
        result["embeddings"]
    )
])

print(f"저장 위치: {DB_PATH}")
print(f"저장된 문서: {stored_count}건")
print(f"임베딩 차원: {len(result['embeddings'][0])}")
print("ChromaDB 재조회 및 적재 검증 성공")

display(check_df.head(2))

저장 위치: C:\Users\Playdata\Desktop\mle-01-p3-team1\data\vector_db\museum_chroma
저장된 문서: 738건
임베딩 차원: 1536
ChromaDB 재조회 및 적재 검증 성공


,museum_id,name,text,embedding_dim
0,M0001,박진전쟁기념관,박진전쟁기념관은 6·25전쟁 최후의 보루였던 박진전투 현장을 재현하여 민족 분단의 ...,1536
1,M0003,창녕박물관,창녕박물관은 1996년 창녕지역 고분군에서 출토된 유물들을 전시할 목적으로 창녕유물...,1536


### step 6 - 검색으로 테스트 해보기

In [10]:
# ==== ChromaDB 의미 검색 테스트 (Top 3) ====

# 1. 사용자 질문 입력
query = input("검색할 질문을 입력하세요: ").strip()

if not query:
    raise ValueError("검색 질문을 입력해야 합니다.")

# 2. 질문 임베딩
query_vector = embeddings.embed_query(query)

# 3. ChromaDB 유사도 검색
results = check_collection.query(
    query_embeddings=[query_vector],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

# 4. 원래 질문 출력
print("\n" + "=" * 70)
print(f"🔎 원래 질문: {query}")
print("=" * 70)

# 5. 상위 3개 검색 결과 출력
for i, (museum_id, document, metadata, distance) in enumerate(
    zip(
        results["ids"][0],
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    metadata = metadata or {}

    print(f"\n{'=' * 70}")
    print(f"📌 검색 결과 {i}위")
    print(f"{'=' * 70}")

    print(f"박물관명 : {metadata.get('name', '정보 없음')}")
    print(f"박물관 ID: {museum_id}")
    print(f"지역     : {metadata.get('sido', '')} {metadata.get('sigungu', '')}")
    print(f"벡터 거리: {distance:.4f}")

    print("\n[검색된 원본 텍스트]")
    print(document)

print(f"\n✅ 총 {len(results['ids'][0])}개 검색 완료")


🔎 원래 질문: 일제강점기 독립운동과 관련된 박물관

📌 검색 결과 1위
박물관명 : 몽양기념관
박물관 ID: M0875
지역     : 경기도 양평군
벡터 거리: 0.9219

[검색된 원본 텍스트]
2011년 11월 27일에 개관. 독립 ·평화통일운동가 여운형 관련 역사박물관

📌 검색 결과 2위
박물관명 : 경상북도독립운동기념관
박물관 ID: M0191
지역     : 경상북도 안동시
벡터 거리: 1.0089

[검색된 원본 텍스트]
경상북도독립운동기념관은 2007년 개관한 안동독립운동기념관을 2014년에 경상북도 전체의 기념관으로 승격·확장하여 2017년 6월 30일 다시 문을 열었다. 경북독립운동사에 관한 자료를 수집‧보존, 조사‧연구, 전시, 교육함으로써 민족의 자주독립을 지켜온 독립운동가들의 나라 사랑 정신을 계승하고 민족문화의 정체성 확립 및 올바른 국가관을 정립하는데 목적을 두고 있다. 또한 경상북도독립운동기념관의 전시실은 독립관과 의열관으로 구성되어 있으며, 1894년 갑오의병부터 1945년 조국 광복 때까지 51년간 펼쳐진 경북 사람들의 국내외 독립운동이 담겨 있다. 의병항쟁을 시작으로 국채보상운동, 자정순국, 만주지역 항일투쟁, 6·10 만세운동, 의열투쟁, 한국광복군 등 경북 사람들의 쉼 없는 항일투쟁과 관련 유물이 소개되어 있다. 또한 의열관은 안동 독립운동의 뿌리가 된 전통마을의 항일투쟁을 한눈에 볼 수 있다. 안동 독립운동의 뿌리가 된 전통마을의 항일투쟁을 전시하고 있는 전시실과 되살아나는 불씨, 나라 위해 목숨 바친 경북 사람들의 51년 독립운동을 영상으로 보여주는 추강영상실, 유아를 위한 교육과 체험공간인 새싹교육실 등으로 구성되어 있다. 경북 지역 독립운동의 역사를 기록하고 기념하는 시설이다. 의병 활동에서 광복에 이르는 지역의 항일운동과 독립운동가들의 활동을 전시·교육을 통해 소개한다.

📌 검색 결과 3위
박물관명 : 국립낙동강생물자원관
박물관 ID: M0519
지역     : 경상북도 상주시
벡터 거리: 1.0385

